In [1]:
import sys
print(sys.executable)

/Users/Maddalena/Documents/color-in-motion/venv/bin/python


In [1]:
import urllib.request
import os

os.makedirs("../data/raw", exist_ok=True)

files = ["title.basics.tsv.gz", "title.ratings.tsv.gz"]
for f in files:
    url = f"https://datasets.imdbws.com/{f}"
    dest = f"../data/raw/{f}"
    if not os.path.exists(dest):
        print(f"Scarico {f}...")
        urllib.request.urlretrieve(url, dest)
        print(f"  fatto")
    else:
        print(f"{f} gia presente")

Scarico title.basics.tsv.gz...
  fatto
Scarico title.ratings.tsv.gz...
  fatto


In [2]:
import pandas as pd

basics = pd.read_csv(
    "../data/raw/title.basics.tsv.gz",
    sep="\t", na_values="\\N", low_memory=False,
    dtype={"startYear": "str", "runtimeMinutes": "str"}
)
print("Righe totali in basics:", len(basics))
basics.head()

Righe totali in basics: 12676776


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,NaN,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,NaN,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,NaN,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,NaN,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,NaN,1,Short


In [3]:
# Carica i voti
ratings = pd.read_csv(
    "../data/raw/title.ratings.tsv.gz",
    sep="\t", na_values="\\N"
)
print("Righe in ratings:", len(ratings))

# Tieni solo i FILM, con anno e generi presenti
film = basics[
    (basics["titleType"] == "movie")
    & (basics["startYear"].notna())
    & (basics["genres"].notna())
].copy()
print("Film (prima dei voti):", len(film))

# Unisci i voti e converti l'anno in numero
film = film.merge(ratings, on="tconst")
film["startYear"] = film["startYear"].astype(int)
print("Film con voti:", len(film))

film[["primaryTitle", "startYear", "genres", "averageRating", "numVotes"]].head()

Righe in ratings: 1700186
Film (prima dei voti): 568952
Film con voti: 336800


,primaryTitle,startYear,genres,averageRating,numVotes
0,Miss Jerry,1894,Romance,5.3,234
1,The Corbett-Fitzsimmons Fight,1897,"Documentary,News,Sport",5.3,606
2,The Story of the Kelly Gang,1906,"Action,Adventure,Biography",6.0,1081
3,The Prodigal Son,1907,Drama,4.8,40
4,Robbery Under Arms,1907,Drama,3.4,36


In [4]:
# Solo film con almeno 10.000 voti
film = film[film["numVotes"] >= 10000]
print("Film rimasti:", len(film))

# Genere principale = primo genere della lista
film["genere_principale"] = film["genres"].str.split(",").str[0]

# Classifica dei generi
film["genere_principale"].value_counts()

Film rimasti: 12489


genere_principale
Comedy         3084
Action         3038
Drama          2469
Crime          1075
Adventure      1003
Biography       759
Horror          653
Documentary     143
Animation        89
Fantasy          77
Mystery          54
Sci-Fi           14
Thriller         14
Romance           7
Family            4
Western           2
Music             1
Musical           1
Film-Noir         1
History           1
Name: count, dtype: int64

In [5]:
generi_scelti = ["Comedy", "Action", "Drama", "Crime", "Horror"]

# Teniamo solo i film di questi cinque generi
film = film[film["genere_principale"].isin(generi_scelti)]

# Per ogni genere prendiamo gli 80 piu votati
campione = film.groupby("genere_principale").apply(
    lambda g: g.nlargest(80, "numVotes")
).reset_index(drop=True)

print("Totale film nel campione:", len(campione))
campione["genere_principale"].value_counts()

Totale film nel campione: 400


/var/folders/ly/h_kts9d54z92kt0g13ljg0ch0000gp/T/ipykernel_8482/712364008.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  campione = film.groupby("genere_principale").apply(


genere_principale
Action    80
Comedy    80
Crime     80
Drama     80
Horror    80
Name: count, dtype: int64

In [6]:
campione.to_csv("../data/processed/campione_film.csv", index=False)
print("Salvato!")

Salvato!


In [10]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
api_key = os.getenv("TMDB_API_KEY")

# Controllo: stampiamo solo i primi caratteri, non tutta la chiave
print("Chiave caricata:", api_key[:6], "...")

Chiave caricata: cecade ...


In [8]:
import os
print("Esiste ../.env ?", os.path.exists("../.env"))

Esiste ../.env ? False


In [9]:
import os
print(os.listdir(".."))

['fase0-result.png', 'fase0.py', 'requirements.txt', 'README.md', '.gitignore', 'test-trailer.mp4', 'venv', '.git', 'data', 'outputs', 'notebooks', 'src']


In [11]:
import requests

# Prendiamo un tconst qualsiasi dal campione per fare la prova
esempio_tconst = campione.iloc[0]["tconst"]
print("Provo con:", esempio_tconst)

# 1. Trova il film su TMDB partendo dall'ID IMDb
url_find = f"https://api.themoviedb.org/3/find/{esempio_tconst}"
params = {"api_key": api_key, "external_source": "imdb_id"}
r = requests.get(url_find, params=params)
risultati = r.json()["movie_results"]

if risultati:
    tmdb_id = risultati[0]["id"]
    print("ID TMDB trovato:", tmdb_id)
else:
    print("Film non trovato su TMDB")

/Users/Maddalena/Documents/color-in-motion/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Provo con: tt0133093
ID TMDB trovato: 603


In [12]:
# 2. Chiediamo i video di questo film
url_videos = f"https://api.themoviedb.org/3/movie/{tmdb_id}/videos"
r = requests.get(url_videos, params={"api_key": api_key})
video = r.json()["results"]

# Cerchiamo il primo trailer su YouTube
link_trailer = None
for v in video:
    if v["type"] == "Trailer" and v["site"] == "YouTube":
        link_trailer = "https://www.youtube.com/watch?v=" + v["key"]
        break

print("Trailer:", link_trailer)

Trailer: https://www.youtube.com/watch?v=FVI84Dfx2-I


In [14]:
def get_trailer_link(tconst):
    # 1. Trova il film su TMDB dall'ID IMDb
    url_find = f"https://api.themoviedb.org/3/find/{tconst}"
    params = {"api_key": api_key, "external_source": "imdb_id"}
    r = requests.get(url_find, params=params)
    risultati = r.json()["movie_results"]

    if not risultati:
        return None  # film non trovato su TMDB

    tmdb_id = risultati[0]["id"]

    # 2. Chiedi i video e cerca il trailer YouTube
    url_videos = f"https://api.themoviedb.org/3/movie/{tmdb_id}/videos"
    r = requests.get(url_videos, params={"api_key": api_key})
    video = r.json()["results"]

    for v in video:
        if v["type"] == "Trailer" and v["site"] == "YouTube":
            return "https://www.youtube.com/watch?v=" + v["key"]

    return None  # nessun trailer trovato

In [15]:
print(get_trailer_link("tt0133093"))

https://www.youtube.com/watch?v=FVI84Dfx2-I


In [16]:
import time

link_trailer = []   # qui salviamo un link per ogni film

for i, riga in campione.iterrows():
    tconst = riga["tconst"]
    link = get_trailer_link(tconst)
    link_trailer.append(link)

    # ogni 50 film stampiamo a che punto siamo
    if i % 50 == 0:
        print(f"Fatti {i} film...")

    time.sleep(0.25)   # piccola pausa tra una chiamata e l'altra

# aggiungiamo la colonna dei link al campione
campione["trailer_link"] = link_trailer
print("Finito!")

Fatti 0 film...
Fatti 50 film...
Fatti 100 film...
Fatti 150 film...
Fatti 200 film...
Fatti 250 film...
Fatti 300 film...
Fatti 350 film...
Finito!


In [17]:
# Quanti film hanno un trailer e quanti no
con_trailer = campione["trailer_link"].notna().sum()
senza_trailer = campione["trailer_link"].isna().sum()

print("Film con trailer:", con_trailer)
print("Film senza trailer:", senza_trailer)

# Quanti trailer per genere
campione[campione["trailer_link"].notna()]["genere_principale"].value_counts()

Film con trailer: 399
Film senza trailer: 1


genere_principale
Action    80
Comedy    80
Crime     80
Horror    80
Drama     79
Name: count, dtype: int64

In [18]:
# Teniamo solo i film che hanno un trailer
campione_finale = campione[campione["trailer_link"].notna()].copy()

# Salviamo la source table completa
campione_finale.to_csv("../data/processed/film_with_trailers.csv", index=False)
print("Salvato:", len(campione_finale), "film")

Salvato: 399 film
